In [37]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

In [38]:
# Configurations

pd.set_option('display.max_columns',None)
pd.set_option('display.float_format',lambda x:f"{x: .3f}")
sns.set_theme(style='whitegrid')
plt.rcParams.update({
    "axes.titlesize":10,
    "axes.labelsize":9,
    "xtick.labelsize":8,
    "ytick.labelsize":8,
})

RANDOM_STATE=42
DATASET_PATH="../data/Delivery_data.csv"
TARGET_COLUMN="delayed"

In [39]:
df=pd.read_csv(DATASET_PATH)

In [40]:
df.head()

,delivery_id,delivery_partner,package_type,vehicle_type,delivery_mode,region,weather_condition,distance_km,package_weight_kg,delivery_time_hours,expected_time_hours,delayed,delivery_status,delivery_rating,delivery_cost
0,250.990,delhivery,automobile parts,bike,same day,west,clear,297.000,46.960,1970-01-01 00:00:00.000000008,1970-01-01 00:00:00.000000008,no,delivered,3,1632.721
1,250.990,xpressbees,cosmetics,ev van,express,central,cold,89.600,47.390,1970-01-01 00:00:00.000000002,1970-01-01 00:00:00.000000003,no,delivered,5,640.170
2,250.990,shadowfax,groceries,truck,two day,east,rainy,273.500,26.890,1970-01-01 00:00:00.000000010,1970-01-01 00:00:00.000000016,no,delivered,4,1448.170
3,250.990,dhl,electronics,ev van,same day,east,cold,269.700,12.690,1970-01-01 00:00:00.000000006,1970-01-01 00:00:00.000000008,no,delivered,3,1486.570
4,250.990,dhl,clothing,van,two day,north,foggy,256.700,37.020,1970-01-01 00:00:00.000000009,1970-01-01 00:00:00.000000016,no,delivered,4,1394.560


In [41]:
df.columns

Index(['delivery_id', 'delivery_partner', 'package_type', 'vehicle_type',
       'delivery_mode', 'region', 'weather_condition', 'distance_km',
       'package_weight_kg', 'delivery_time_hours', 'expected_time_hours',
       'delayed', 'delivery_status', 'delivery_rating', 'delivery_cost'],
      dtype='str')

In [42]:
df.shape

(25000, 15)

In [43]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   delivery_id          25000 non-null  float64
 1   delivery_partner     25000 non-null  str    
 2   package_type         25000 non-null  str    
 3   vehicle_type         25000 non-null  str    
 4   delivery_mode        25000 non-null  str    
 5   region               25000 non-null  str    
 6   weather_condition    25000 non-null  str    
 7   distance_km          25000 non-null  float64
 8   package_weight_kg    25000 non-null  float64
 9   delivery_time_hours  25000 non-null  str    
 10  expected_time_hours  25000 non-null  str    
 11  delayed              25000 non-null  str    
 12  delivery_status      25000 non-null  str    
 13  delivery_rating      25000 non-null  int64  
 14  delivery_cost        25000 non-null  float64
dtypes: float64(4), int64(1), str(10)
memory usage: 

In [44]:
df.isna().sum()

delivery_id            0
delivery_partner       0
package_type           0
vehicle_type           0
delivery_mode          0
region                 0
weather_condition      0
distance_km            0
package_weight_kg      0
delivery_time_hours    0
expected_time_hours    0
delayed                0
delivery_status        0
delivery_rating        0
delivery_cost          0
dtype: int64

In [45]:
df['delayed'].value_counts()

delayed
no     18331
yes     6669
Name: count, dtype: int64

In [46]:
df['delivery_status'].value_counts()

delivery_status
delivered    18331
delayed       5341
failed        1328
Name: count, dtype: int64

In [47]:
df["delivery_time_hours"].value_counts().head(20)

delivery_time_hours
1970-01-01 00:00:00.000000006    3118
1970-01-01 00:00:00.000000005    3011
1970-01-01 00:00:00.000000007    2921
1970-01-01 00:00:00.000000004    2571
1970-01-01 00:00:00.000000008    2386
1970-01-01 00:00:00.000000003    2352
1970-01-01 00:00:00.000000009    1875
1970-01-01 00:00:00.000000002    1668
1970-01-01 00:00:00.000000010    1371
1970-01-01 00:00:00.000000011    1039
1970-01-01 00:00:00.000000001     963
1970-01-01 00:00:00.000000012     649
1970-01-01 00:00:00.000000013     371
1970-01-01 00:00:00.000000000     255
1970-01-01 00:00:00.000000014     247
1970-01-01 00:00:00.000000015     116
1970-01-01 00:00:00.000000016      51
1970-01-01 00:00:00.000000017      26
1970-01-01 00:00:00.000000018       7
1970-01-01 00:00:00.000000019       3
Name: count, dtype: int64

In [48]:
df["expected_time_hours"].value_counts().head(20)

expected_time_hours
1970-01-01 00:00:00.000000016    6302
1970-01-01 00:00:00.000000008    6287
1970-01-01 00:00:00.000000024    6186
1970-01-01 00:00:00.000000005    1080
1970-01-01 00:00:00.000000007    1051
1970-01-01 00:00:00.000000004    1037
1970-01-01 00:00:00.000000003    1029
1970-01-01 00:00:00.000000002    1015
1970-01-01 00:00:00.000000006    1013
Name: count, dtype: int64

In [49]:
df['delivery_time_hours']=pd.to_datetime(df['delivery_time_hours']).dt.nanosecond
df['expected_time_hours']=pd.to_datetime(df['expected_time_hours']).dt.nanosecond

In [50]:
df[[
    "delivery_time_hours",
    "expected_time_hours",
    "delayed"
]].head(20)

,delivery_time_hours,expected_time_hours,delayed
0,8,8,no
1,2,3,no
2,10,16,no
3,6,8,no
4,9,16,no
5,4,2,yes
6,6,8,no
7,4,8,no
8,5,8,no
9,3,8,no


In [51]:
df= df.drop(columns=['delivery_id','delivery_status','expected_time_hours','delivery_time_hours','delivery_rating'])

In [52]:
df

,delivery_partner,package_type,vehicle_type,delivery_mode,region,weather_condition,distance_km,package_weight_kg,delayed,delivery_cost
0,delhivery,automobile parts,bike,same day,west,clear,297.000,46.960,no,1632.721
1,xpressbees,cosmetics,ev van,express,central,cold,89.600,47.390,no,640.170
2,shadowfax,groceries,truck,two day,east,rainy,273.500,26.890,no,1448.170
3,dhl,electronics,ev van,same day,east,cold,269.700,12.690,no,1486.570
4,dhl,clothing,van,two day,north,foggy,256.700,37.020,no,1394.560
...,...,...,...,...,...,...,...,...,...,...
24995,dhl,furniture,scooter,two day,east,foggy,80.700,3.650,no,414.450
24996,ecom express,automobile parts,ev van,standard,west,clear,172.900,21.420,no,928.760
24997,ekart,clothing,scooter,same day,east,rainy,168.400,4.850,yes,956.550
24998,shadowfax,documents,ev van,standard,west,stormy,37.200,8.040,no,210.120


In [56]:
X= df.drop(columns=['delayed'])
y=df[TARGET_COLUMN]

0         no
1         no
2         no
3         no
4         no
        ... 
24995     no
24996     no
24997    yes
24998     no
24999     no
Name: delayed, Length: 25000, dtype: str